## **Diagram “Current-State Siloed GRC Data Landscape” (As-Is Silo Model)**

```mermaid
flowchart LR
    A["Business & Operations<br>• Customer Data (CRM)<br>• Onboarding data<br>• Profile & KYC"]:::one
    B["IT / Data Management<br>• Transaction Logs<br>• Payment Systems<br>• Channels & Devices"]:::one
    C["Risk & Compliance<br>• AML / CFT Data<br>• Credit Card Risk<br>• Alerts & Flags"]:::two
    D["Internal Audit<br>• Limited / delayed access<br>• Broken data pipeline<br>• Incomplete visibility"]:::three

    A -->|Siloed, partial sharing| D
    B -->|Restricted, request-based| D
    C -->|Filtered, anonymized| D

    classDef one fill:#e8f4ff,stroke:#0066cc
    classDef two fill:#fff5cc,stroke:#cc9900
    classDef three fill:#fde2e2,stroke:#cc3333
```

## **The Diagram Where Silos Process are happen**

``` mermaid

flowchart TB
    subgraph S1["Data Silo #1: Customer"]
        A1["• CRM / Onboarding\n• KYC / Profile\n• Owned by Business"]
    end

    subgraph S2["Data Silo #2: Transaction"]
        A2["• Core Banking\n• Transaction Logs\n• Device / IP / Channel\n• Owned by IT/Ops"]
    end

    subgraph S3["Data Silo #3: Compliance/AML"]
        A3["• AML Alerts\n• Fraud Flags (PaySim)\n• Risk Scoring\n• Owned by Compliance & Risk"]
    end

    IA["Internal Audit\n• Fragmented Access\n• Delayed Data\n• Partial View Only"]

    S1 --> IA
    S2 --> IA
    S3 --> IA

```


## **Diagram “How Data Should Flow Ideally (3LOD Integrated)”**

```mermaid

flowchart LR
    A["First Line (Business, Ops, IT)\n• Operational data"]:::one
    B["Second Line (Risk, Compliance)\n• Oversight data"]:::two
    C["Third Line (Internal Audit)\n• Independent assurance"]:::three

    D["Unified Data Layer\n(CUSTOMER_MASTER_FULL + TRANSACTION_MASTER_FULL)"]:::data

    A -->|operational data| D
    B -->|risk controls & monitoring| D
    C -->|audit insights| D

    D -->|data access| A
    D -->|risk intelligence| B
    D -->|assurance evidence| C

    classDef one fill:#e6f7ff,stroke:#3399ff
    classDef two fill:#fff6d9,stroke:#cc9900
    classDef three fill:#ffe6e6,stroke:#cc3333
    classDef data fill:#e8ffe6,stroke:#22aa22

```

# **Before vs After – Siloed vs Integrated GRC Data**

## **As-Is: Siloed GRC Data Landscape**

```mermaid

flowchart LR
    subgraph LINE1["First Line - Business & Ops"]
        CUST["Customer Data\n(CRM, onboarding)"]
        TRX["Transaction Data\n(Core banking, payment systems)"]
        PROD["Product / Channel Data"]
    end

    subgraph LINE2["Second Line - Risk & Compliance"]
        RISK["Risk & Compliance Systems\n(AML, sanctions, KYC)"]
    end

    subgraph LINE3["Third Line - Internal Audit"]
        IA["Internal Audit\n(Ad-hoc data requests,\nExcel extracts, reports)"]
    end

    CUST --> RISK
    TRX --> RISK

    RISK --> IA
    CUST -. partial / delayed data .-> IA
    TRX -. partial / sample data .-> IA


```

## **To-Be: Integrated GRC Data Landscape (Conceptual)**

```mermaid

flowchart LR
    subgraph LINE1I["First Line - Business & Ops"]
        CUSTI["Operational Systems\n(CRM, Core Banking, Channels)"]
    end

    subgraph LINE2I["Second Line - Risk & Compliance"]
        RISKI["Risk & Compliance\n(AML, KYC, Sanctions, Fraud)"]
    end

    subgraph LINE3I["Third Line - Internal Audit"]
        IAI["Internal Audit\n(Continuous / Thematic Reviews)"]
    end

    HUB["Unified Data Layer\n(Synthetic + Aggregated GRC Data)"]

    CUSTI --> HUB
    RISKI --> HUB

    HUB --> CUSTI
    HUB --> RISKI
    HUB --> IAI


```

## **3LOD + SDG (Synthetic Data) Architecture**

``` mermaid

flowchart LR

    A["First Line (Business, Ops, IT)<br>• Operational data"]:::one
    B["Second Line (Risk, Compliance)<br>• Oversight data"]:::two
    C["Third Line (Internal Audit)<br>• Independent assurance"]:::three

    D["Synthetic Data Layer<br>CUSTOMER_MASTER_FULL<br>TRANSACTION_MASTER_FULL"]:::data
    E["SDG / ETL Pipelines<br>Faker, routing logic,<br>feature engineering"]:::etl

    A -->|operational data| E
    B -->|policies and controls| E

    E --> D

    D -->|KRI and portfolio views| B
    D -->|process views and journeys| A
    D -->|evidence for testing| C

    C -->|assurance and recommendations| A
    C -->|control effectiveness feedback| B

    classDef one fill:#e6f7ff,stroke:#3399ff
    classDef two fill:#fff6d9,stroke:#cc9900
    classDef three fill:#ffe6e6,stroke:#cc3333
    classDef data fill:#e8ffe6,stroke:#22aa22
    classDef etl fill:#f2e8ff,stroke:#8844cc



```

## **DATA PIPELINE (Customer + Transaction + Credit Card → Synthetic Data)**

``` mermaid

flowchart TB
    %% CUSTOMER PIPELINE
    subgraph CUST_PIPE["Customer Data Pipeline"]
        C1["churn.csv"] --> C_STD["Standardize & Rename"]
        C2["bank_additional_full.csv"] --> C_STD
        C3["marketing_campaign.csv"] --> C_STD

        C_STD --> C_ALIGN["Resample to align\nrow counts"]
        C_ALIGN --> C_MERGE["Merge into unified\ncustomer dataframe"]
        C_MERGE --> C_FAKE["Faker enrichment\n(first_name,\nfallback surname)"]
        C_FAKE --> C_FEAT["Derive date_of_birth\nfrom age"]
        C_FEAT --> C_SEL["Select final schema"]
        C_SEL --> C_OUT["CUSTOMER_MASTER_FULL.csv"]
    end

    %% TRANSACTION PIPELINE
    subgraph TRX_PIPE["Transaction Data Pipeline"]
        T1["bank_transactions_data_2.csv"] --> T_STD["Standardize & Rename\n(transaction fields)"]
        T_STD --> T_BEH["Behavioral features\n(prev_tx_date,\nlogin_attempts,\nduration)"]
        C_OUT --> T_JOIN["Join with\nCUSTOMER_MASTER_FULL"]
        T_BEH --> T_JOIN

        T_JOIN --> T_ACCT["Generate accounts\n(account_id, type,\naccount_created_date,\ncurrency)"]
        T_ACCT --> T_ROUTE["Generate routing\n(sender/receiver\ncountry, region,\nsender_account)"]
        T_ROUTE --> T_SEL["Select final schema"]
        T_SEL --> T_OUT["TRANSACTION_MASTER_FULL.csv"]
    end


```

## **Simulation: First Line**

In [4]:
import pandas as pd
import numpy as np

In [5]:
customer = pd.read_csv(r"D:\KULIAH\SEMESTER 4\DRAFT\Step by step\Final Source Data\CUSTOMER_MASTER_FULL.csv")
transaction = pd.read_csv(r"D:\KULIAH\SEMESTER 4\DRAFT\Step by step\Final Source Data\TRANSACTION_MASTER_FULL.csv")

In [6]:
customer.head()

,customer_id,first_name,surname,gender,date_of_birth,age,marital,education,job,income,contact,geography,balance,has_cr_card,default,housing,loan,credit_score,tenure,dt_customer
0,C2540,Patrick,Hargrave,Female,1983-03-17,42,SINGLE,MASTER,self-employed,66726.0,LANDLINE,France,0.00,1,NO,NO,NO,619,11,2014-01-12
1,C1455,Jonathan,Hill,Female,1984-02-10,41,DIVORCED,UNIVERSITY,technician,41883.0,MOBILE,Spain,83807.86,0,UNKNOWN,YES,NO,608,12,2013-03-19
2,C2701,Jennifer,Onio,Female,1983-01-28,42,SINGLE,UNIVERSITY,student,43824.0,MOBILE,France,159660.80,1,NO,NO,YES,502,13,2012-09-15
3,C5074,James,Boni,Female,1986-10-11,39,MARRIED,PHD,admin.,65640.0,MOBILE,France,0.00,0,UNKNOWN,YES,NO,699,11,2014-03-01
4,C7320,Heather,Mitchell,Female,1982-06-03,43,SINGLE,UNIVERSITY,technician,48686.0,LANDLINE,Spain,125510.82,1,NO,YES,NO,850,12,2013-12-04


In [7]:
customer.shape

(6121, 20)

In [8]:
customer.columns

Index(['customer_id', 'first_name', 'surname', 'gender', 'date_of_birth',
       'age', 'marital', 'education', 'job', 'income', 'contact', 'geography',
       'balance', 'has_cr_card', 'default', 'housing', 'loan', 'credit_score',
       'tenure', 'dt_customer'],
      dtype='object')

In [9]:
customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6121 entries, 0 to 6120
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customer_id    6121 non-null   object 
 1   first_name     6121 non-null   object 
 2   surname        6121 non-null   object 
 3   gender         6121 non-null   object 
 4   date_of_birth  6121 non-null   object 
 5   age            6121 non-null   int64  
 6   marital        6121 non-null   object 
 7   education      6121 non-null   object 
 8   job            6121 non-null   object 
 9   income         6064 non-null   float64
 10  contact        6121 non-null   object 
 11  geography      6121 non-null   object 
 12  balance        6121 non-null   float64
 13  has_cr_card    6121 non-null   int64  
 14  default        6121 non-null   object 
 15  housing        6121 non-null   object 
 16  loan           6121 non-null   object 
 17  credit_score   6121 non-null   int64  
 18  tenure  

In [10]:
customer.describe()

,age,income,balance,has_cr_card,credit_score,tenure
count,6121.000000,6064.000000,6121.000000,6121.000000,6121.000000,6121.000000
mean,38.894462,51855.831464,76260.255087,0.705604,650.592223,11.971410
std,10.451802,22827.839361,62563.687125,0.455808,97.758157,0.685517
min,18.000000,2447.000000,0.000000,0.000000,350.000000,11.000000
25%,32.000000,34916.000000,0.000000,0.000000,583.000000,12.000000
50%,37.000000,51369.000000,96863.520000,1.000000,652.000000,12.000000
75%,44.000000,68118.000000,127510.990000,1.000000,718.000000,12.000000
max,92.000000,666666.000000,250898.090000,1.000000,850.000000,13.000000


In [11]:
customer.describe(include=[object])

,customer_id,first_name,surname,gender,date_of_birth,marital,education,job,contact,geography,default,housing,loan,dt_customer
count,6121,6121,6121,6121,6121,6121,6121,6121,6121,6121,6121,6121,6121,6121
unique,6121,615,2239,2,4720,4,5,12,2,3,2,3,3,654
top,C2540,Michael,Fanucci,Male,1988-02-22,MARRIED,UNIVERSITY,admin.,MOBILE,France,NO,YES,NO,2012-09-12
freq,1,137,19,3335,5,3669,3054,1593,3853,3058,4901,3213,5004,36


In [12]:
print(customer.isnull().sum())
print(customer.duplicated().sum())

customer_id       0
first_name        0
surname           0
gender            0
date_of_birth     0
age               0
marital           0
education         0
job               0
income           57
contact           0
geography         0
balance           0
has_cr_card       0
default           0
housing           0
loan              0
credit_score      0
tenure            0
dt_customer       0
dtype: int64
0


In [13]:
transaction.head()

,transaction_id,transaction_date,transaction_amount,transaction_type,transaction_duration,login_attempts,previous_transaction_date,previous_transaction_date_raw,location_device,device_id,...,dt_customer,account_id,account_type,currency,account_created_date,sender_account,sender_country,sender_region,receiver_country,receiver_region
0,TX001994,2023-04-25 18:47:16,568.68,Debit,101,1,2014-05-29 00:00:00,2024-11-04 08:08:39,Houston,D000283,...,2014-05-29,EE123211568472519237,BUSINESS,EUR,2014-05-29,EE998098223737835459,Mexico,America,Estonia,Europe
1,TX001283,2023-07-12 16:58:17,76.02,Credit,189,1,2023-04-25 18:47:16,2024-11-04 08:11:51,Albuquerque,D000644,...,2014-05-29,EE123211568472519237,BUSINESS,EUR,2014-05-29,EE275276046906451786,South Korea,Asia,Estonia,Europe
2,TX001608,2023-10-24 16:12:19,122.81,Debit,12,1,2023-07-12 16:58:17,2024-11-04 08:09:37,San Jose,D000419,...,2014-05-29,EE123211568472519237,BUSINESS,EUR,2014-05-29,EE134304038606410374,Italy,Europe,Estonia,Europe
3,TX001948,2023-11-29 16:27:38,354.96,Debit,96,1,2023-10-24 16:12:19,2024-11-04 08:08:19,Fort Worth,D000282,...,2014-05-29,EE123211568472519237,BUSINESS,EUR,2014-05-29,EE661804901436589210,Spain,Europe,Estonia,Europe
4,TX001353,2023-12-28 16:34:52,192.23,Debit,39,3,2023-11-29 16:27:38,2024-11-04 08:06:41,Albuquerque,D000117,...,2014-05-29,EE123211568472519237,BUSINESS,EUR,2014-05-29,EE274608981976978886,Denmark,Europe,Estonia,Europe


In [14]:
transaction.shape

(2512, 26)

In [15]:
transaction.columns

Index(['transaction_id', 'transaction_date', 'transaction_amount',
       'transaction_type', 'transaction_duration', 'login_attempts',
       'previous_transaction_date', 'previous_transaction_date_raw',
       'location_device', 'device_id', 'ip_address', 'channel', 'location',
       'customer_id', 'first_name', 'surname', 'dt_customer', 'account_id',
       'account_type', 'currency', 'account_created_date', 'sender_account',
       'sender_country', 'sender_region', 'receiver_country',
       'receiver_region'],
      dtype='object')

In [16]:
transaction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2512 entries, 0 to 2511
Data columns (total 26 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   transaction_id                 2512 non-null   object 
 1   transaction_date               2512 non-null   object 
 2   transaction_amount             2512 non-null   float64
 3   transaction_type               2512 non-null   object 
 4   transaction_duration           2512 non-null   int64  
 5   login_attempts                 2512 non-null   int64  
 6   previous_transaction_date      2512 non-null   object 
 7   previous_transaction_date_raw  2512 non-null   object 
 8   location_device                2512 non-null   object 
 9   device_id                      2512 non-null   object 
 10  ip_address                     2512 non-null   object 
 11  channel                        2512 non-null   object 
 12  location                       2512 non-null   o

In [17]:
transaction.describe()

,transaction_amount,transaction_duration,login_attempts
count,2512.000000,2512.000000,2512.000000
mean,297.593778,119.643312,1.124602
std,291.946243,69.963757,0.602662
min,0.260000,10.000000,1.000000
25%,81.885000,63.000000,1.000000
50%,211.140000,112.500000,1.000000
75%,414.527500,161.000000,1.000000
max,1919.110000,300.000000,5.000000


In [18]:
transaction.describe(include=[object])

,transaction_id,transaction_date,transaction_type,previous_transaction_date,previous_transaction_date_raw,location_device,device_id,ip_address,channel,location,...,dt_customer,account_id,account_type,currency,account_created_date,sender_account,sender_country,sender_region,receiver_country,receiver_region
count,2512,2512,2512,2512,2512,2512,2512,2512,2512,2512,...,2512,2512,2512,2512,2512,2512,2512,2512,2512,2512
unique,2512,2512,2,2352,360,43,681,592,3,39,...,335,495,3,2,335,1358,28,5,1,1
top,TX001994,2023-04-25 18:47:16,Debit,2014-02-10 00:00:00,2024-11-04 08:09:17,Fort Worth,D000315,200.136.146.93,BRANCH,Rakvere,...,2013-02-12,EE759570867709678296,CURRENT,EUR,2013-02-12,EE967799834684623822,Thailand,Europe,Estonia,Europe
freq,1,1,1944,5,16,70,9,13,868,258,...,25,12,880,1321,25,10,102,1004,2512,2512


In [19]:
transaction.describe(include=[object])

,transaction_id,transaction_date,transaction_type,previous_transaction_date,previous_transaction_date_raw,location_device,device_id,ip_address,channel,location,...,dt_customer,account_id,account_type,currency,account_created_date,sender_account,sender_country,sender_region,receiver_country,receiver_region
count,2512,2512,2512,2512,2512,2512,2512,2512,2512,2512,...,2512,2512,2512,2512,2512,2512,2512,2512,2512,2512
unique,2512,2512,2,2352,360,43,681,592,3,39,...,335,495,3,2,335,1358,28,5,1,1
top,TX001994,2023-04-25 18:47:16,Debit,2014-02-10 00:00:00,2024-11-04 08:09:17,Fort Worth,D000315,200.136.146.93,BRANCH,Rakvere,...,2013-02-12,EE759570867709678296,CURRENT,EUR,2013-02-12,EE967799834684623822,Thailand,Europe,Estonia,Europe
freq,1,1,1944,5,16,70,9,13,868,258,...,25,12,880,1321,25,10,102,1004,2512,2512


## **AS-IS Process Simulation in the Banking Industry**

This step is about simulation, it helps to illustrate the business process in banking industry where business functions operate in silos. By simulating the GRC & Audit process as a part of 3LOD to show how  a single customer case flows across the First, Second, and Third Lines of Defence in a traditional banking environment. It is based on the bank’s customer master data and transaction master datato show how one customer’s journey creates fragmented information across IT & Operations, Marketing, Sales, Risk, Compliance, and Audit.
The simulation connects each function to the Three Lines of Defence (1st, 2nd, and 3rd line) to demonstrate misalignment and data gaps in the GRC & Audit process. 


## **Simulation Theme**

Transaction Anomaly (Cross-border High-value Transaction)


## **Simulation: First Line**

### **IT & Operation Team - Detecting and Processing Transactions Anomaly/volume**

**What they see:**
Operations & IT see the raw transaction as it enters the system.

**Data used:**
transaction_amount, transaction_date, transaction_type, sender_country, receiver_country, account_balance, device_id, ip_address, login_attempts, previous_transaction_date

**AS-IS Behaviour:**

1. The core banking system flags the transaction as unusual (large inbound amount from another country), but:

2. The alert is only visible locally in the transaction system.

3. IT system logs (device_id, IP address, login attempts) are stored in a separate log server.

Operations export only partial fields to compliance (date, amount, sender_country).

**→ Business impact:**
The First Line detects the anomaly but does not escalate it properly because their report excludes device and behaviour anomalies.

Output to Second Line: partial transaction data → missing critical indicators.

## **Simulation: Second Line**

## **Risk & Compliance Team - Reviewing the Same Transaction Anomaly**

**What they receive:**

Only a truncated dataset from Operations.

**Data used:**
transaction_amount, sender_country, receiver_country, channel, customer_id

**customer profile:
income, balance, credit_score, geography (nationality), loan**

**AS-IS Behaviour:**

1. Risk & Compliance identify the transaction as questionable:

2. high-value vs customer income

3. cross-border from monitored region

customer nationality mismatches transaction origin

no economic rationale


also, They do NOT receive:

- device_id

- ip_address

- login_attempts

- transaction velocity behaviour

- device mismatch history

Compliance requests the missing fields, but IT responds days later.

Because evidence is incomplete, Compliance documents the case as:

**→ “Unusual but not suspicious due to insufficient data.”**

**Output to Third Line: compliance decision + incomplete case file.**

## **Simulation: Third Line**

### **Audit Team - Validating the Handling of the Same Transaction Anomaly**

**What they attempt to do:**
Audit wants to reconstruct the exact customer journey.

**Data needed:**

1. full transaction (from Operations)

2. full customer profile

3. risk scoring (from Risk team)

4. AML/KYC screening details (from Compliance)

5. device/IP logs (from IT)

AS-IS Behaviour:

Audit discovers:

- IT is unable to provide complete logs (privacy restrictions).

- Operations only provide partial data — same extract used for compliance.

- Compliance cannot justify closing the case because evidence was incomplete.

- No unified system to reconcile customer → transaction → device logging.

**Audit issues a major finding:**

“Transaction anomalies cannot be assessed reliably because First Line and Second Line operate with fragmented datasets. Controls are ineffective.”

→ Output: audit finding on fragmented GRC process.



## **Transaction Anomaly: high-value inbound cross-border transfer**

| Line                           | Activity                                  | Visibility                                | Result                     |
| ------------------------------ | ----------------------------------------- | ----------------------------------------- | -------------------------- |
| **1st Line** (Ops/IT)          | Processes anomaly, creates partial report | Sees full logs but shares incomplete data | Alert not escalated        |
| **2nd Line** (Risk/Compliance) | Reviews anomaly using incomplete data     | No device/IP/login info                   | Case closed incorrectly    |
| **3rd Line** (Audit)           | Tries to validate if process was correct  | Cannot obtain full evidence               | Issues major audit finding |


In [20]:


# -----------------------------------------------------------
# PARSE DATETIME COLUMNS (AFTER LOADING DATA)
# -----------------------------------------------------------

date_cols_txn = [
    "transaction_date",
    "previous_transaction_date",
    "previous_transaction_date_raw",
    "account_created_date"
]

for col in date_cols_txn:
    if col in transaction.columns:
        transaction[col] = pd.to_datetime(transaction[col], errors="coerce")

if "dt_customer" in customer.columns:
    customer["dt_customer"] = pd.to_datetime(customer["dt_customer"], errors="coerce")

# -----------------------------------------------------------
# SELECT ONE ANOMALY CASE (DEMO)
# -----------------------------------------------------------

anomaly_case = transaction[
    (transaction["transaction_amount"] > transaction["transaction_amount"].median()) &
    (transaction["sender_country"] != transaction["receiver_country"])
].head(1)

if anomaly_case.empty:
    anomaly_case = transaction.sample(1)

txn_id = anomaly_case["transaction_id"].iloc[0]
print("Using transaction anomaly case:", txn_id)

# -----------------------------------------------------------
# 1ST LINE – OPERATIONS (PARTIAL REPORT)
# -----------------------------------------------------------

first_line_cols = [
    "transaction_id",
    "customer_id",
    "transaction_amount",
    "transaction_date",
    "transaction_type",
    "sender_country",
    "receiver_country",
    "channel",
    "account_id"
]

first_line_cols = [c for c in first_line_cols if c in transaction.columns]
first_line_view = anomaly_case[first_line_cols].copy()
first_line_view["line"] = "FIRST_LINE"
first_line_view.to_csv("ASIS_FIRST_LINE_VIEW.csv", index=False)

# -----------------------------------------------------------
# 2ND LINE – COMPLIANCE (INCOMPLETE DATA)
# -----------------------------------------------------------

compliance_view = first_line_view.merge(
    customer,
    on="customer_id",
    how="left",
    suffixes=("_txn", "_cust")
)

# Basic AML rule with LIMITED data (AS-IS)
def basic_aml_flag(row):
    amount = row.get("transaction_amount", np.nan)
    income = row.get("income", np.nan)
    sender_country = str(row.get("sender_country", ""))

    if pd.isna(amount) or pd.isna(income) or income <= 0:
        return False

    high_value = amount > 3 * income
    cross_border = sender_country != row.get("receiver_country")
    return high_value and cross_border

compliance_view["basic_suspicious_flag"] = compliance_view.apply(basic_aml_flag, axis=1)

compliance_view["compliance_decision"] = np.where(
    compliance_view["basic_suspicious_flag"],
    "UNUSUAL_NOT_SUSPICIOUS",
    "NORMAL"
)

compliance_view["line"] = "SECOND_LINE"
compliance_view.to_csv("ASIS_SECOND_LINE_VIEW.csv", index=False)

# -----------------------------------------------------------
# 3RD LINE – INTERNAL AUDIT (FULL DATA)
# -----------------------------------------------------------

full_txn = transaction[transaction["transaction_id"] == txn_id]
audit_view = full_txn.merge(
    customer,
    on="customer_id",
    how="left"
)

def full_aml_flag(row):
    amount = row.get("transaction_amount", np.nan)
    income = row.get("income", np.nan)
    cross_border = row.get("sender_country") != row.get("receiver_country")
    high_value = amount > 3 * income

    # Audit also sees behavioural indicators
    login_attempts = row.get("login_attempts", 0)
    many_login_attempts = (login_attempts >= 3)

    return (high_value and cross_border) or many_login_attempts

audit_view["audit_suspicious_flag"] = audit_view.apply(full_aml_flag, axis=1)
audit_view["line"] = "THIRD_LINE"
audit_view.to_csv("ASIS_THIRD_LINE_VIEW.csv", index=False)

# -----------------------------------------------------------
# COMPARE SECOND LINE VS AUDIT FINDINGS
# -----------------------------------------------------------

comparison = audit_view[[
    "transaction_id", "transaction_amount", "sender_country", "receiver_country",
    "login_attempts", "audit_suspicious_flag"
]].merge(
    compliance_view[[
        "transaction_id", "basic_suspicious_flag", "compliance_decision"
    ]],
    on="transaction_id",
    how="left"
)

comparison.to_csv(r"D:\KULIAH\SEMESTER 4\DRAFT\Step by step\Final Source Data\ASIS_ANALYSIS_GAPS.csv", index=False)

print("\n=== AS-IS SIMULATION COMPLETE ===")
print("Generated the following files:")
print("  - ASIS_FIRST_LINE_VIEW.csv")
print("  - ASIS_SECOND_LINE_VIEW.csv")
print("  - ASIS_THIRD_LINE_VIEW.csv")
print("  - ASIS_ANALYSIS_GAPS.csv")


Using transaction anomaly case: TX001994

=== AS-IS SIMULATION COMPLETE ===
Generated the following files:
  - ASIS_FIRST_LINE_VIEW.csv
  - ASIS_SECOND_LINE_VIEW.csv
  - ASIS_THIRD_LINE_VIEW.csv
  - ASIS_ANALYSIS_GAPS.csv


## **The Diagram Where Silos Process are happen**

```mermaid

flowchart TB

    %% ----------------------------
    %% SILO 1 — CUSTOMER DOMAIN
    %% ----------------------------
    subgraph S1 [Data Silo 1 - Customer Domain]
        A1["CRM / Onboarding\nKYC / Profile Data\nIncome / Credit Score\nOwned by Business"]
    end

    %% ----------------------------
    %% SILO 2 — TRANSACTION DOMAIN
    %% ----------------------------
    subgraph S2 [Data Silo 2 - Transactions & Channels]
        A2["Core Banking Transactions\nDevice / IP Logs\nLogin Attempts\nChannel Activity\nOwned by IT/Ops"]
    end

    %% ----------------------------
    %% SILO 3 — COMPLIANCE / RISK DOMAIN
    %% ----------------------------
    subgraph S3 [Data Silo 3 - AML and Fraud]
        A3["AML Alerts\nFraud Flags\nSanctions Screening\nRisk Scoring\nOwned by Compliance & Risk"]
    end

    %% ----------------------------
    %% INTERNAL AUDIT
    %% ----------------------------
    IA["Internal Audit\nFragmented Access\nDelayed Requests\nNo Unified Evidence\nPartial View"]

    %% ----------------------------
    %% BROKEN FLOWS
    %% ----------------------------
    S1 -->|"Partial customer data\n(no behaviour context)"| S3
    S2 -->|"Limited logs shared\n(device/IP not included)"| S3

    S1 -->|"Ad-hoc extract"| IA
    S2 -->|"Request-based logs"| IA
    S3 -->|"Filtered summary cases"| IA



```

## **DIAGRAM FOR FRAUD/AML USE CASE**

```mermaid
flowchart LR

    %% ===========================
    %% FIRST LINE
    %% ===========================
    subgraph S1 [Customer Data Silo - First Line]
        A1["KYC Profile\nCustomer Data\nIncome and Credit Score\nOwned by Business"]
    end

    subgraph S2 [Transaction and Behaviour Silo - First Line]
        A2["Core Banking Transactions\nDevice Logs\nIP Logs\nLogin Attempts\nOwned by IT and Operations"]
    end

    %% ===========================
    %% SECOND LINE
    %% ===========================
    subgraph S3 [AML and Fraud Silo - Second Line]
        A3["AML Alerts\nFraud Flags\nSanctions Screening\nRisk Scoring\nOwned by Compliance and Risk"]
    end

    %% ===========================
    %% THIRD LINE
    %% ===========================
    IA["Internal Audit - Third Line\nFragmented Access\nDelayed Data\nPartial Evidence"]

    %% ===========================
    %% BROKEN FLOWS
    %% ===========================
    S1 -->|"Partial customer data"| S3
    S2 -->|"Missing behaviour logs"| S3

    S1 -->|"Ad hoc extracts"| IA
    S2 -->|"Request based logs"| IA
    S3 -->|"Filtered case summaries"| IA


```

## **Ideal Diagram For Data Flow in The 3LOD Integrated**

```mermaid
flowchart LR

    A["First Line (Business, Ops, IT)
    Full operational data
    Controls and process execution"]:::one

    B["Second Line (Risk, Compliance)
    Monitoring and oversight data
    Risk assessment and AML/Fraud review"]:::two

    C["Third Line (Internal Audit)
    Independent assurance
    Control and data validation"]:::three

    %% FLOWS
    A -->|"Share complete operational data"| B
    B -->|"Oversight and risk escalation"| A

    A -->|"Provide full and timely data"| C
    B -->|"Provide oversight records"| C

    C -->|"Assurance, findings, recommendations"| A
    C -->|"Feedback and risk awareness"| B

    %% COLORS
    classDef one fill:#e6f7ff,stroke:#3399ff,stroke-width:1px;
    classDef two fill:#fff6d9,stroke:#cc9900,stroke-width:1px;
    classDef three fill:#ffe6e6,stroke:#cc3333,stroke-width:1px;


```

We simulate one suspicious transaction that appears to be both a **fraud anomaly and a potential money laundering pattern**. This same transaction travels through the **1st, 2nd, and 3rd line**, demonstrating data fragmentation and ineffective GRC coordination.

## **DATA PIPELINE**

```mermaid

flowchart TB

    %% RAW DATA
    subgraph RAW [RAW DATA]
        C1[churn.csv]
        C2[bank_additional_full.csv]
        C3[marketing_campaign.tsv]
        T1[bank_transactions_data_2.csv]
        F1[faker enrichment]
    end

    %% CUSTOMER PIPELINE
    subgraph CUST_PIPE [CUSTOMER PIPELINE]
        C_STD[Standardize Customer Fields]
        C_MERGE[Merge Customer Sources]
        C_CLEAN[Clean and Deduplicate]
        C_MASTER[CUSTOMER_MASTER_FULL.csv]
    end

    C1 --> C_STD
    C2 --> C_STD
    C3 --> C_STD
    F1 --> C_STD

    C_STD --> C_MERGE
    C_MERGE --> C_CLEAN
    C_CLEAN --> C_MASTER

    %% TRANSACTION PIPELINE
    subgraph TRX_PIPE [TRANSACTION PIPELINE]
        T_STD[Standardize Transactions]
        T_FE[Feature Engineering]
        T_JOIN[Join with Customer Master]
        T_MASTER[TRANSACTION_MASTER_FULL.csv]
    end

    T1 --> T_STD
    T_STD --> T_FE
    T_FE --> T_JOIN
    C_MASTER --> T_JOIN
    T_JOIN --> T_MASTER

    %% OUTPUT
    subgraph OUTPUT [GRC AND AUDIT USE]
        OUT[Unified Dataset for AML Fraud Audit]
    end

    C_MASTER --> OUT
    T_MASTER --> OUT

```

## **1st LINE OF DEFENCE – Operations & IT**

**Activity: Detect and Process a Suspicious Transaction**

**What the 1st line sees (full raw data from system logs):**

**From Transaction Master:**

transaction_amount

sender_country, receiver_country

transaction_type

account_id, account_balance

device_id, ip_address

login_attempts

transaction_date

transaction_duration

previous_transaction_date

**From Customer Master:**

customer_id, income, balance, credit_score, geography

tenure, has_cr_card, loan

**What happens AS-IS:**

1. The system detects that a customer receives a high-value transfer from a foreign country.

2. The mobile banking logs show 3–5 failed login attempts and suspicious device/IP changes.

3. Operations exports a daily transaction report for Compliance—but:

**Reason: These fields sit in separate IT subsystems (mobile logs, security logs, fraud engine).**

**Output from 1st Line:**

A partial transaction report → missing fraud/AML evidence.

## **2nd LINE OF DEFENCE – Risk & Compliance**

**Activity: Fraud Detection + AML Review Using Incomplete Data**

**What the 2nd line receives:**
Only the 1st-line partial report + customer information.

**Available from joined dataset:**

transaction_amount

transaction_date

sender_country / receiver_country

income

balance

credit_score

customer nationality (geography)

**Fraud Team perspective (Risk):**

- Sees high-value inbound transfer

- Sees cross-border risk

- But cannot confirm account takeover → because they don’t receive login_attempts or device_id.

**Compliance (AML/KYC) perspective:**

- Flags mismatch:

    - income vs transaction_amount

    - nationality vs sender_country

    - Wants to check behavioural evidence → must request manually from IT.

But…

- Logs arrive days later

- Some logs cannot be matched due to different system identifiers

- Evidence incomplete

**AS-IS Decision:**

“Unusual but not suspicious due to lack of supporting log evidence.”

**No STR filed.**

**Fraud detection + AML fails.**

## **3rd LINE OF DEFENCE – Internal Audit**

**Activity: Validate Fraud/AML Handling of the Same Case**

**Audit attempts to gather:**

- full transaction data

- full customer data

- compliance case files

- IT security logs (device_id, login_attempts, IP)

- fraud engine alerts

**AS-IS Reality:**

- Operations re-sends the same partial dataset that Compliance used

- IT logs are:

    - delayed

    - incomplete

    - not linked to the transaction ID

- Compliance only provides summary notes, not evidence

- Fraud engine logs cannot be reconciled because of siloed identifiers

**Audit Findings:**

1, Compliance decision incorrect
→ If audit uses the full dataset (including device/IP logs), the case is suspicious.

2. Fraud detection ineffective
→ Risk team did not see login/IP anomalies due to data fragmentation.

3. System fragmentation across all lines:

    - Core banking
        ↔ digital log system
        ↔ fraud engine
        ↔ AML system
        ↔ customer master

**Audit Final Conclusion:**

“Money laundering and fraud risks cannot be reliably assessed because First and Second Line rely on partial and inconsistent data extracts. This represents a major GRC failure.”

| Line         | What They Do                          | What They Miss                                        |
| ------------ | ------------------------------------- | ----------------------------------------------------- |
| **1st Line** | Processes transaction, sees full logs | Sends partial data → crucial fraud/AML fields missing |
| **2nd Line** | Fraud & AML analysis                  | Cannot escalate → no device/IP/login data             |
| **3rd Line** | Audit reconstructs case               | Confirms failure → issues major finding               |


## **1ST LINE CODE — OPERATIONS & IT**


**Business processes the transaction + partial report creation**


In [21]:
# Pick anomaly: large cross-border transaction
anomaly_case = transaction[
    (transaction["transaction_amount"] > transaction["transaction_amount"].median()) &
    (transaction["sender_country"] != transaction["receiver_country"])
].head(1)

if anomaly_case.empty:
    anomaly_case = transaction.sample(1)

txn_id = anomaly_case["transaction_id"].iloc[0]
print("Selected Transaction for Scenario:", txn_id)

# Only fields sent to Compliance (AS-IS)
first_line_cols = [
    "transaction_id", "customer_id", "transaction_amount", "transaction_date",
    "transaction_type", "sender_country", "receiver_country", "channel",
    "account_id"
]

first_line_cols = [c for c in first_line_cols if c in anomaly_case.columns]

first_line_report = anomaly_case[first_line_cols].copy()
first_line_report["line"] = "FIRST_LINE"

first_line_report.to_csv("ASIS_1ST_LINE_REPORT.csv", index=False)
first_line_report


Selected Transaction for Scenario: TX001994


,transaction_id,customer_id,transaction_amount,transaction_date,transaction_type,sender_country,receiver_country,channel,account_id,line
0,TX001994,C1001,568.68,2023-04-25 18:47:16,Debit,Mexico,Estonia,ATM,EE123211568472519237,FIRST_LINE


## **2ND LINE CODE — RISK & COMPLIANCE**

**Fraud detection & AML review using incomplete data**

In [22]:
# Join partial Ops data with customer master
second_line_view = first_line_report.merge(
    customer, on="customer_id", how="left"
)

# BASIC AML RULE (because data incomplete)
def second_line_flag(row):
    amt = row["transaction_amount"]
    inc = row.get("income", np.nan)
    cross = (row["sender_country"] != row["receiver_country"])
    
    if pd.isna(amt) or pd.isna(inc) or inc <= 0:
        return False

    return (amt > 3 * inc) and cross

second_line_view["basic_suspicious_flag"] = second_line_view.apply(second_line_flag, axis=1)

# AS-IS DECISION: insufficient evidence → false negative
second_line_view["decision_2nd_line"] = np.where(
    second_line_view["basic_suspicious_flag"],
    "UNUSUAL_NOT_SUSPICIOUS",
    "NORMAL"
)

second_line_view["line"] = "SECOND_LINE"

second_line_view.to_csv("ASIS_2ND_LINE_REPORT.csv", index=False)
second_line_view


,transaction_id,customer_id,transaction_amount,transaction_date,transaction_type,sender_country,receiver_country,channel,account_id,line,...,balance,has_cr_card,default,housing,loan,credit_score,tenure,dt_customer,basic_suspicious_flag,decision_2nd_line
0,TX001994,C1001,568.68,2023-04-25 18:47:16,Debit,Mexico,Estonia,ATM,EE123211568472519237,SECOND_LINE,...,0.0,1,UNKNOWN,YES,NO,598,11,2014-05-29,False,NORMAL


## **3RD LINE CODE — INTERNAL AUDIT**

**Audit sees full logs → identifies failure in 1st & 2nd line**

In [23]:
audit_view = transaction[transaction["transaction_id"] == txn_id].merge(
    customer, on="customer_id", how="left"
)

# FULL AUDIT RULE (audit has all data)
def audit_flag(row):
    amt = row["transaction_amount"]
    inc = row.get("income", np.nan)
    cross = (row["sender_country"] != row["receiver_country"])
    login_attempts = row.get("login_attempts", 0)
    
    if pd.isna(amt) or pd.isna(inc) or inc <= 0:
        return False
    
    high_value = amt > 3 * inc
    many_logins = login_attempts >= 3

    # audit sees both financial + behavioural patterns
    return (high_value and cross) or many_logins

audit_view["audit_suspicious_flag"] = audit_view.apply(audit_flag, axis=1)
audit_view["line"] = "THIRD_LINE"

audit_view.to_csv("ASIS_3RD_LINE_REPORT.csv", index=False)
audit_view


,transaction_id,transaction_date,transaction_amount,transaction_type,transaction_duration,login_attempts,previous_transaction_date,previous_transaction_date_raw,location_device,device_id,...,balance,has_cr_card,default,housing,loan,credit_score,tenure,dt_customer_y,audit_suspicious_flag,line
0,TX001994,2023-04-25 18:47:16,568.68,Debit,101,1,2014-05-29,2024-11-04 08:08:39,Houston,D000283,...,0.0,1,UNKNOWN,YES,NO,598,11,2014-05-29,False,THIRD_LINE
